In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [20]:
import pandas as pd
import numpy as np

In [21]:
def load_raw_data(path='/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'):
    train = pd.read_csv(path + 'train.csv.zip', parse_dates=['Date'])
    test = pd.read_csv(path + 'test.csv.zip', parse_dates=['Date'])
    features = pd.read_csv(path + 'features.csv.zip', parse_dates=['Date'])
    stores = pd.read_csv(path + 'stores.csv')
    return train, test, features, stores

train, test, features, stores = load_raw_data()

In [22]:
def merge_all(df, features, stores):
    df = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    df = df.merge(stores, on='Store', how='left')
    return df

In [23]:
def combine_train_test(train, test, features, stores):
    train_df = merge_all(train, features, stores)
    test_df = merge_all(test, features, stores)

    train_df['is_train'] = 1
    test_df['is_train'] = 0
    test_df['Weekly_Sales'] = np.nan

    full_df = pd.concat([train_df, test_df], ignore_index=True)
    full_df = full_df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return full_df

In [24]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

def process_markdowns(df):
    df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
    df['markdown_total'] = df[MARKDOWN_COLS].sum(axis=1)
    df['n_active_markdowns'] = (df[MARKDOWN_COLS] > 0).sum(axis=1)
    return df

In [25]:
def flag_sparse_series(df, train_df, min_weeks=52):
    pair_counts = train_df.groupby(['Store', 'Dept'])['Date'].count().reset_index(name='n_weeks')
    df = df.merge(pair_counts, on=['Store', 'Dept'], how='left')
    df['is_sparse'] = df['n_weeks'] < min_weeks
    return df

In [26]:
def add_calendar_features(df):
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['month'] = df['Date'].dt.month
    df['year'] = df['Date'].dt.year

    major_holidays = pd.to_datetime([
        '2010-02-12', '2011-02-11', '2012-02-10',  # Super Bowl
        '2010-09-10', '2011-09-09', '2012-09-07',  # Labor Day
        '2010-11-26', '2011-11-25', '2012-11-23',  # Thanksgiving
        '2010-12-31', '2011-12-30', '2012-12-28',  # Christmas (week bucket)
    ])

    df['days_to_nearest_holiday'] = df['Date'].apply(
        lambda d: min(abs((d - h).days) for h in major_holidays)
    )

    thanksgiving_dates = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23'])
    week_after = [d + pd.Timedelta(weeks=1) for d in thanksgiving_dates]
    df['is_week_after_thanksgiving'] = df['Date'].isin(week_after).astype(int)

    return df

In [27]:
def add_lag_rolling_features(df):
    df = df.sort_values(['Store', 'Dept', 'Date'])
    group = df.groupby(['Store', 'Dept'])['Weekly_Sales']

    for lag in [1, 2, 3, 4]:
        df[f'lag_{lag}'] = group.shift(lag)

    df['lag_52'] = group.shift(52)
    df['lag_53'] = group.shift(53)

    shifted = group.shift(1)
    df['rolling_mean_4'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(4).mean())
    df['rolling_mean_8'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(8).mean())
    df['rolling_std_4'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(4).std())
    df['rolling_std_8'] = shifted.groupby([df['Store'], df['Dept']]).transform(lambda x: x.rolling(8).std())

    df['yoy_growth'] = (df['lag_1'] - df['lag_53']) / df['lag_53'].replace(0, np.nan)

    return df

In [28]:
def add_group_aggregates(df, train_df):
    dept_avg = train_df.groupby('Dept')['Weekly_Sales'].mean().rename('dept_avg_sales')
    store_avg = train_df.groupby('Store')['Weekly_Sales'].mean().rename('store_avg_sales')

    df = df.merge(dept_avg, on='Dept', how='left')
    df = df.merge(store_avg, on='Store', how='left')
    return df

In [29]:
def add_store_type_ordinal(df):
    type_order = {'A': 3, 'B': 2, 'C': 1}
    df['type_ordinal'] = df['Type'].map(type_order)
    return df

In [30]:
def run_feature_pipeline(train, test, features, stores, min_weeks=52):
    full_df = combine_train_test(train, test, features, stores)
    full_df = process_markdowns(full_df)
    full_df = flag_sparse_series(full_df, train, min_weeks=min_weeks)
    full_df = add_calendar_features(full_df)
    full_df = add_lag_rolling_features(full_df)
    full_df = add_group_aggregates(full_df, train)
    full_df = add_store_type_ordinal(full_df)
    return full_df

full_df = run_feature_pipeline(train, test, features, stores)
print(full_df.shape)
full_df.head()

(536634, 40)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,lag_52,lag_53,rolling_mean_4,rolling_mean_8,rolling_std_4,rolling_std_8,yoy_growth,dept_avg_sales,store_avg_sales,type_ordinal
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,...,NaN,NaN,32990.77,NaN,12832.106391,NaN,NaN,19213.485088,21710.543621,3


In [32]:
def split_and_save(full_df):
    processed_train = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
    processed_test = full_df[full_df['is_train'] == 0].drop(columns=['is_train', 'Weekly_Sales'])

    processed_train.to_csv('processed_train.csv', index=False)
    processed_test.to_csv('processed_test.csv', index=False)

    print("Train:", processed_train.shape, " Test:", processed_test.shape)
    return processed_train, processed_test

processed_train, processed_test = split_and_save(full_df)

Train: (421570, 39)  Test: (115064, 38)
